## RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\Dell\AppData\Local\Temp\ipykernel_15324\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [3]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file)) 
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 5 PDF files to process

Processing: EE3355_L01_4392.pdf
  ✓ Loaded 15 pages

Processing: EE3355_L03_4392.pdf
  ✓ Loaded 3 pages

Processing: EE3355_L1_4385.pdf
  ✓ Loaded 12 pages

Processing: Lab 3.pdf
  ✓ Loaded 12 pages

Processing: Lab_01 Report Format.pdf
  ✓ Loaded 17 pages

Total documents loaded: 59


In [4]:
len(all_pdf_documents)


59

In [5]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=400,chunk_overlap=50):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [6]:
chunks=split_documents(all_pdf_documents)
chunks

Split 59 documents into 96 chunks

Example chunk:
Content: EE3355: SIGNAL AND SYSTEM 
LABORATORY 01 
 
 
 
 
 
 
 
 
 
 
 
 
                                                                        NAME        : AKARSHANA A.N.  
                               ...
Metadata: {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-11-07T12:35:32+05:30', 'author': 'DEIE', 'moddate': '2026-11-07T12:35:32+05:30', 'source': '..\\data\\pdf\\EE3355_L01_4392.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'EE3355_L01_4392.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-11-07T12:35:32+05:30', 'author': 'DEIE', 'moddate': '2026-11-07T12:35:32+05:30', 'source': '..\\data\\pdf\\EE3355_L01_4392.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'EE3355_L01_4392.pdf', 'file_type': 'pdf'}, page_content='EE3355: SIGNAL AND SYSTEM \nLABORATORY 01 \n \n \n \n \n \n \n \n \n \n \n \n \n                                                                        NAME        : AKARSHANA A.N.  \n                                                                        REG No.     : EE/2021/4392 \n                SEMESTER: 03 \n                                                                        DATE          : 20/11 /2026'),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-11-07T12:35:32+05:30', 'author': 'DEIE', 'moddate': '2026-11-07T12:35:32+05:30', 'source': '..\\data\\pd

## Embedding And VectorStoreDB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


c:\Nimesh_Temp\HiLangChain\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dell\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Nimesh_Temp\HiLangChain\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarni

Model loaded successfully. Embedding dimension: 384


C:\Users\Dell\AppData\Local\Temp\ipykernel_15324\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


## VectoreStore

In [9]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [10]:
chunks

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-11-07T12:35:32+05:30', 'author': 'DEIE', 'moddate': '2026-11-07T12:35:32+05:30', 'source': '..\\data\\pdf\\EE3355_L01_4392.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'EE3355_L01_4392.pdf', 'file_type': 'pdf'}, page_content='EE3355: SIGNAL AND SYSTEM \nLABORATORY 01 \n \n \n \n \n \n \n \n \n \n \n \n \n                                                                        NAME        : AKARSHANA A.N.  \n                                                                        REG No.     : EE/2021/4392 \n                SEMESTER: 03 \n                                                                        DATE          : 20/11 /2026'),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-11-07T12:35:32+05:30', 'author': 'DEIE', 'moddate': '2026-11-07T12:35:32+05:30', 'source': '..\\data\\pd

In [11]:
len(chunks)

96

In [34]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]
texts

['EE3355: SIGNAL AND SYSTEM \nLABORATORY 01 \n \n \n \n \n \n \n \n \n \n \n \n \n                                                                        NAME        : AKARSHANA A.N.  \n                                                                        REG No.     : EE/2021/4392 \n                SEMESTER: 03 \n                                                                        DATE          : 20/11 /2026',
 'Table 1.1: Summative Laboratory Form \nSemester 03 \nModule Code EE3355 \nModule Name Signal and system \nLab Number 01 \nLab Name Continuous-time Signal Analysis \nLab conduction date 2026.11.04 \nReport Submission date 2026.11.20',
 'Contents \n1 Observation 6 \n1.1 Part 01 Observe the waveform in Time domain and Frequency domain 6 \n▪ Amplitude-5.12V 6 \n1.2 Square waveform 8 \n 8 \n1.2 Part 02: Observe the Lissajous pattern using oscilloscope while comparing frequency and the \nphase shift of the different waveform. 11 \n2 Calculations 14 \n2.1 Specimen calculations 1

In [13]:
## Generate the Embeddings
embeddings=embedding_manager.generate_embeddings(texts)

Generating embeddings for 96 texts...


Batches: 100%|██████████| 3/3 [00:26<00:00,  8.78s/it]

Generated embeddings with shape: (96, 384)


In [25]:
embeddings[95]

array([ 1.12870848e-02,  5.43413348e-02,  4.53727208e-02, -1.41788542e-01,
       -1.24716284e-02,  4.37368117e-02,  1.27055675e-01,  7.80145004e-02,
        4.99545550e-03, -3.75079252e-02, -9.15484577e-02, -1.23586893e-01,
       -1.03041260e-02,  7.26923868e-02,  2.04975344e-02,  5.87498099e-02,
       -2.69593112e-02, -8.72188658e-02, -6.85322955e-02,  1.99395567e-02,
       -1.38012618e-02, -4.05836143e-02, -1.77583508e-02, -7.36605795e-03,
       -5.69184981e-02, -2.56785378e-02, -1.81911066e-02,  1.51055504e-03,
        1.92459505e-02,  4.63029109e-02,  3.16610150e-02,  3.94124873e-02,
        3.70722599e-02, -7.60693699e-02,  1.57949533e-02, -1.15311079e-01,
        1.75808445e-02, -4.96852994e-02, -1.85185019e-02, -2.19994411e-02,
       -5.49913161e-02, -4.47978117e-02,  7.63408691e-02, -3.90467606e-02,
       -2.10481416e-02, -5.32268472e-02, -1.56124989e-02, -5.24836294e-02,
        1.30105644e-01, -5.94255403e-02, -5.58009744e-02, -2.27259509e-02,
        7.83802662e-03, -

In [26]:
##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Adding 96 documents to vector store...
Successfully added 96 documents to vector store
Total documents in collection: 96


### Retriever Pipeline From VectorStore

In [27]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [28]:
rag_retriever

In [38]:
rag_retriever.retrieve("What is observed?")

Retrieving documents for query: 'What is observed?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.90s/it]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

In [ ]:
rag_retriever.retrieve(" Time domain square waveform-10% DutyObserved cycle")

Retrieving documents for query: 'Observed Time domain square waveform-10% Duty cycle'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.15s/it]


Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_68e9650f_10',
  'content': 'Figure 7 : Observed Time domain square waveform-90% Duty cycle',
  'metadata': {'page_label': '10',
   'creator': 'Microsoft® Word 2016',
   'page': 9,
   'creationdate': '2026-11-07T12:35:32+05:30',
   'doc_index': 10,
   'source': '..\\data\\pdf\\EE3355_L01_4392.pdf',
   'author': 'DEIE',
   'moddate': '2026-11-07T12:35:32+05:30',
   'file_type': 'pdf',
   'content_length': 62,
   'producer': 'Microsoft® Word 2016',
   'total_pages': 15,
   'source_file': 'EE3355_L01_4392.pdf'},
  'similarity_score': 0.8861396312713623,
  'distance': 0.1138603687286377,
  'rank': 1},
 {'id': 'doc_93a4bf8e_9',
  'content': 'Figure 5: Observed Time domain square waveform-10% Duty cycle \n \n \n \n \n \n \n \n \n \n \n \n \n \n                   Figure 6 : Observed Frequency domain waveform-10% Duty cycle',
  'metadata': {'moddate': '2026-11-07T12:35:32+05:30',
   'page_label': '9',
   'source': '..\\data\\pdf\\EE3355_L01_4392.pdf',
   'total_pages': 15,
   'page

### RAG Pipeline- VectorDB To LLM Output Generation

In [42]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [44]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [45]:
class GroqLLM:
    def __init__(self, model_name: str = "openai/gpt-oss-120b", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [46]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: openai/gpt-oss-120b
Groq LLM initialized successfully!


In [47]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("Time domain square waveform-10% DutyObserved cycle")

Retrieving documents for query: 'Time domain square waveform-10% DutyObserved cycle'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.49s/it]


Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_68e9650f_10',
  'content': 'Figure 7 : Observed Time domain square waveform-90% Duty cycle',
  'metadata': {'source': '..\\data\\pdf\\EE3355_L01_4392.pdf',
   'moddate': '2026-11-07T12:35:32+05:30',
   'creationdate': '2026-11-07T12:35:32+05:30',
   'creator': 'Microsoft® Word 2016',
   'file_type': 'pdf',
   'total_pages': 15,
   'producer': 'Microsoft® Word 2016',
   'page_label': '10',
   'page': 9,
   'content_length': 62,
   'source_file': 'EE3355_L01_4392.pdf',
   'author': 'DEIE',
   'doc_index': 10},
  'similarity_score': 0.7344389259815216,
  'distance': 0.2655610740184784,
  'rank': 1},
 {'id': 'doc_93a4bf8e_9',
  'content': 'Figure 5: Observed Time domain square waveform-10% Duty cycle \n \n \n \n \n \n \n \n \n \n \n \n \n \n                   Figure 6 : Observed Frequency domain waveform-10% Duty cycle',
  'metadata': {'file_type': 'pdf',
   'producer': 'Microsoft® Word 2016',
   'moddate': '2026-11-07T12:35:32+05:30',
   'page': 8,
   'doc_index': 9,
   'cont

### Integration Vectordb Context pipeline With LLM output

In [50]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [52]:
answer=rag_simple("Time domain square waveform-10% DutyObserved cycle?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'Time domain square waveform-10% DutyObserved cycle?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 13.67it/s]

Generated embeddings with shape: (1, 384)


Retrieved 3 documents (after filtering)
The observed duty cycle of the time‑domain square waveform is **10 %**.


### Enhanced RAG Pipeline Features

# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

In [ ]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])